#  Scaled Dot-Product Attention (PyTorch Implementation)

The **Scaled Dot-Product Attention** is the fundamental building block of the Transformer architecture.  
It computes how much each token should attend to other tokens in the sequence.  


## Formula  

For queries \(Q\), keys \(K\), and values \(V\):  

$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$

- \(Q\): Query matrix  
- \(K\): Key matrix  
- \(V\): Value matrix  
- \(d_k\): Dimensionality of keys (used for scaling)  



##  Step-by-Step Code Explanation

Import PyTorch and required modules.

math.sqrt will be used for scaling.



In [6]:
import torch
import torch.nn.functional as F
import math

## **Define Attention Function**

In [12]:
def dot_product_attention(Q, K, V, scale=True):
    """
    Q: (batch_size, seq_len_q, d_k)
    K: (batch_size, seq_len_k, d_k)
    V: (batch_size, seq_len_v, d_v) where seq_len_v = seq_len_k
    """
    # Step 1: Compute raw attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1))
    # Shape: (batch_size, seq_len_q, seq_len_k)

    # Step 2: Scale the scores (recommended for stability)
    if scale:
        d_k = Q.size(-1)
        scores = scores / math.sqrt(d_k)

    # Step 3: Apply softmax to convert scores into probabilities
    attention_weights = F.softmax(scores, dim=-1)

    # Step 4: Apply attention weights to values
    output = torch.matmul(attention_weights, V)
    # Shape: (batch_size, seq_len_q, d_v)

    return output, attention_weights


step 1:

- We take dot products between each query and all keys.

- This gives similarity scores (how much a query should attend to each key).

Step 2:

- Scaling prevents large dot-product values when d_k is large.

- Helps stabilize gradients.

Step 3:

- Softmax ensures all weights sum to 1.

- Higher score → higher attention weight.

Step 4:

- The weighted sum of values gives the final attended representation.

- Output has the same length as the queries (seq_len_q).

In [13]:
batch_size, seq_len, d_model = 2, 5, 64

Q = torch.randn(batch_size, seq_len, d_model)
K = torch.randn(batch_size, seq_len, d_model)
V = torch.randn(batch_size, seq_len, d_model)

output, weights = dot_product_attention(Q, K, V)

print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")


Output shape: torch.Size([2, 5, 64])
Attention weights shape: torch.Size([2, 5, 5])


**Expected Output Shapes**

- **Outpu**t: (batch_size, seq_len_q, d_v) → (2, 5, 64)

- **Attention Weights**: (batch_size, seq_len_q, seq_len_k) → (2, 5, 5)

This means:

- Each query attends to all 5 keys.

- Attention weights form a probability distribution over keys for each query.

## ***Key Insight***

- **Queries (Q)**: What I’m looking for.

- **Keys (K):** What I contain.

- **Values (V):** What I pass on if attended.

The model learns to **focus on relevant parts of the sequence** dynamically, which is why attention outperforms fixed-context RNNs/LSTMs.

# Additive Attention (Bahdanau Attention)

Additive Attention was introduced by **Bahdanau et al. (2015)** in their neural machine translation paper.  
It computes attention scores by **learning a feed-forward scoring function** instead of using dot products.



## Formula

Given a query \(q\), keys \(k_i\), and values \(v_i\):

$
e_i = v^T \tanh(W_q q + W_k k_i + b)
$

$
\alpha_i = \text{softmax}(e_i)
$

$
\text{context} = \sum_{i=1}^{T} \alpha_i v_i
$

- $(W_q, W_k, v, b$): Learned parameters  
- $(e_i$): Alignment score for key $(k_i$)  
- $(\alpha_i$): Attention weight  
- $(\text{context}$): Weighted sum of values  


## PyTorch Implementation


In [15]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class AdditiveAttention(nn.Module):
    def __init__(self, query_dim, key_dim, attention_dim):
        super(AdditiveAttention, self).__init__()
        self.query_dim = query_dim
        self.key_dim = key_dim
        self.attention_dim = attention_dim

        # Learned projections
        self.W_q = nn.Linear(query_dim, attention_dim, bias=False)
        self.W_k = nn.Linear(key_dim, attention_dim, bias=False)
        self.v = nn.Linear(attention_dim, 1, bias=False)
        self.bias = nn.Parameter(torch.randn(attention_dim))

    def forward(self, query, keys, values):
        """
        query: (batch_size, query_dim)
        keys: (batch_size, seq_len, key_dim)
        values: (batch_size, seq_len, value_dim)
        """
        batch_size, seq_len, _ = keys.size()

        # Step 1: Expand query and project query/keys
        query_expanded = query.unsqueeze(1).expand(-1, seq_len, -1)
        q_proj = self.W_q(query_expanded)  # (batch_size, seq_len, attention_dim)
        k_proj = self.W_k(keys)            # (batch_size, seq_len, attention_dim)

        # Step 2: Non-linear combination (tanh)
        combined = torch.tanh(q_proj + k_proj + self.bias)

        # Step 3: Compute scores
        scores = self.v(combined).squeeze(-1)  # (batch_size, seq_len)

        # Step 4: Normalize with softmax
        attention_weights = F.softmax(scores, dim=-1)

        # Step 5: Weighted sum of values
        context = torch.sum(attention_weights.unsqueeze(-1) * values, dim=1)

        return context, attention_weights


**Step 1: Expand and Project**
```python
query_expanded = query.unsqueeze(1).expand(-1, seq_len, -1)
q_proj = self.W_q(query_expanded)
k_proj = self.W_k(keys)
```

- xpand the query to match the sequence length.

- Project query and keys into the same attention space.


**Step 2: Combine with Non-linearity**

```python
combined = torch.tanh(q_proj + k_proj + self.bias)
```
- Add projected query, projected key, and bias.
- Apply tanh to introduce non-linearity.

**Step 3: Compute Raw Scores**
```python
scores = self.v(combined).squeeze(-1)
```
- Use a learned vector
𝑣
v to compute alignment scores.

- Shape: (batch_size, seq_len).

**Step 4: Normalize**

```python
attention_weights = F.softmax(scores, dim=-1)
```
- Convert scores into probabilities across the sequence.

- Each row sums to 1.

**Step 5: Compute Context**
```python
context = torch.sum(attention_weights.unsqueeze(-1) * values, dim=1)
```

- Weighted sum of values using attention weights.

- Final context vector has shape (batch_size, value_dim).

## **Example Usage**

In [ ]:
query_dim, key_dim, value_dim, attention_dim = 256, 512, 512, 128
batch_size, seq_len = 4, 10

attention = AdditiveAttention(query_dim, key_dim, attention_dim)

query = torch.randn(batch_size, query_dim)
keys = torch.randn(batch_size, seq_len, key_dim)
values = torch.randn(batch_size, seq_len, value_dim)

context, weights = attention(query, keys, values)

print(f"Context shape: {context.shape}")          # (4, 512)
print(f"Attention weights shape: {weights.shape}") # (4, 10)


## **Expected Output Shapes**

- **Context**: (batch_size, value_dim) → (4, 512)

- **Attention Weights**: (batch_size, seq_len) → (4, 10)

## **Key Insight**

- Unlike dot-product attention, additive attention uses a feed-forward network to compute scores.

- This makes it more flexible but slightly slower than scaled dot-product attention.

# General Multiplicative Attention (Luong Attention)

General Multiplicative Attention is a variant of **Luong Attention**, where a **learned weight matrix** transforms the keys before computing the dot product with the query.  

This makes the model more flexible than simple dot-product attention.  

#Implementation of General Multiplicative attention

In [16]:

class GeneralMultiplicativeAttention(nn.Module):
    def __init__(self, query_dim, key_dim):
        super(GeneralMultiplicativeAttention, self).__init__()
        self.W_a = nn.Linear(key_dim, query_dim, bias=False)

    def forward(self, query, keys, values):
        """
        query: (batch_size, query_dim)
        keys: (batch_size, seq_len, key_dim)
        values: (batch_size, seq_len, value_dim)
        """
        # Transform keys to match query dimension
        keys_transformed = self.W_a(keys)  # (batch_size, seq_len, query_dim)

        # Compute scores
        scores = torch.sum(query.unsqueeze(1) * keys_transformed, dim=-1)  # (batch_size, seq_len)

        # Apply softmax
        attention_weights = F.softmax(scores, dim=-1)

        # Apply attention
        context = torch.sum(attention_weights.unsqueeze(-1) * values, dim=1)

        return context, attention_weights

## **Step 1: Transform Keys**

```python
keys_transformed = self.W_a(keys)
```
- Apply a learned linear transformation W_a to keys.

- Ensures query and keys have the same dimension.

- Shape: (batch_size, seq_len, query_dim)

## **Step 2: Compute Alignment Scores**
```python
scores = torch.sum(query.unsqueeze(1) * keys_transformed, dim=-1)
```
Expand query to match the sequence length. Compute similarity using element-wise multiplication. Sum across last dimension → gives raw alignment scores.Shape: (batch_size, seq_len)

**Step 3: Normalize with Softmax**

```python
attention_weights = F.softmax(scores, dim=-1)
```
- Convert raw scores into probabilities.

- Each row sums to 1.

- Shape: (batch_size, seq_len)

## **Step 4: Compute Context Vector**

```python
context = torch.sum(attention_weights.unsqueeze(-1) * values, dim=1)
```
- Weight the values using attention probabilities.

- Take weighted sum → gives final context vector.

- Shape: (batch_size, value_dim

**Final Output**

- context → summarizes important information from values.

- attention_weights → shows how much focus is given to each key.

# Concat Multiplicative Attention (Bahdanau-style)

Concat Multiplicative Attention (also known as **Additive Attention**) works by **concatenating query and key vectors**, applying a feed-forward network with a non-linearity, and then using a learned vector to produce attention scores.  






In [ ]:
#Implementation of Concat Multiplicative attention

class ConcatMultiplicativeAttention(nn.Module):
    def __init__(self, query_dim, key_dim, attention_dim):
        super(ConcatMultiplicativeAttention, self).__init__()
        self.W = nn.Linear(query_dim + key_dim, attention_dim, bias=True)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, query, keys, values):
        batch_size, seq_len, _ = keys.size()

        # Expand query to match sequence length
        query_expanded = query.unsqueeze(1).expand(-1, seq_len, -1)

        # Concatenate query and keys
        combined = torch.cat([query_expanded, keys], dim=-1)  # (batch_size, seq_len, query_dim + key_dim)

        # Apply linear transformation and tanh
        hidden = torch.tanh(self.W(combined))  # (batch_size, seq_len, attention_dim)

        # Compute scores
        scores = self.v(hidden).squeeze(-1)  # (batch_size, seq_len)

        # Apply softmax
        attention_weights = F.softmax(scores, dim=-1)

        # Apply attention
        context = torch.sum(attention_weights.unsqueeze(-1) * values, dim=1)

        return context, attention_weights



### **Step 1: Expand Query**
- Expand the **query** so it matches the sequence length of keys.  
- Shape becomes `(batch_size, seq_len, query_dim)`.  



### **Step 2: Concatenate Query and Keys**
- Concatenate expanded **query** and **keys** along the last dimension.  
- Shape: `(batch_size, seq_len, query_dim + key_dim)`.  



### **Step 3: Apply Linear Transformation + Non-linearity**
- Pass concatenated vectors through a linear layer `W` followed by **tanh** activation.  
- This creates a hidden representation in **attention space**.  
- Shape: `(batch_size, seq_len, attention_dim)`.  



### **Step 4: Compute Scores**
- Apply another linear layer `v` to the hidden representation.  
- This produces raw attention scores for each key.  
- Shape: `(batch_size, seq_len)`.  



### **Step 5: Normalize with Softmax**
- Apply **softmax** on scores along the sequence length.  
- Converts raw scores into **attention weights** (probabilities).  
- Shape: `(batch_size, seq_len)` → each row sums to 1.  



### **Step 6: Compute Context Vector**
- Multiply attention weights with the **values**.  
- Take a weighted sum across sequence length.  
- Produces the final **context vector**.  
- Shape: `(batch_size, value_dim)`.  



## Final Output
- **context** → Weighted representation of the most relevant values.  
- **attention_weights** → Distribution showing which keys were most important.  

In [ ]:
## Complete Comparison Example

# Set up data
batch_size, seq_len = 2, 6
query_dim, key_dim, value_dim = 64, 64, 64
attention_dim = 32

query = torch.randn(batch_size, query_dim)
keys = torch.randn(batch_size, seq_len, key_dim)
values = torch.randn(batch_size, seq_len, value_dim)

# Initialize all attention mechanisms
additive_attn = AdditiveAttention(query_dim, key_dim, attention_dim)
general_mult_attn = GeneralMultiplicativeAttention(query_dim, key_dim)
concat_mult_attn = ConcatMultiplicativeAttention(query_dim, key_dim, attention_dim)

# Apply all attention mechanisms
context_add, weights_add = additive_attn(query, keys, values)
context_gen, weights_gen = general_mult_attn(query, keys, values)
context_con, weights_con = concat_mult_attn(query, keys, values)

print("Attention Weights Comparison:")
print(f"Additive: {weights_add[0]}")
print(f"General Multiplicative: {weights_gen[0]}")
print(f"Concat Multiplicative: {weights_con[0]}")

# For dot-product attention (when query_dim == key_dim)
Q = query.unsqueeze(1).expand(-1, seq_len, -1)
K = keys
V = values
context_dot, weights_dot = dot_product_attention(Q, K, V)
print(f"Dot-product: {weights_dot[0, 0]}")  # First query position


Attention Weights Comparison:
Additive: tensor([0.1624, 0.1196, 0.1549, 0.1413, 0.2297, 0.1921],
       grad_fn=<SelectBackward0>)
General Multiplicative: tensor([3.9416e-03, 5.7043e-04, 7.7151e-01, 1.8342e-02, 2.0310e-01, 2.5287e-03],
       grad_fn=<SelectBackward0>)
Concat Multiplicative: tensor([0.2036, 0.1430, 0.1753, 0.1471, 0.1712, 0.1598],
       grad_fn=<SelectBackward0>)
Dot-product: tensor([0.0896, 0.2962, 0.0927, 0.0317, 0.0198, 0.4700])


#  Complete Comparison of Attention Mechanisms

In this example above, we compare **four different attention mechanisms**:  
1. **Additive Attention (Bahdanau Attention)**  
2. **General Multiplicative Attention**  
3. **Concat Multiplicative Attention**  
4. **Dot-Product Attention (Luong / Scaled Dot-Product)**  


### **Step 1: Set up Input Data**
- Define batch size = 2 and sequence length = 6.  
- Create random `query`, `keys`, and `values` with dimension sizes `(64, 64, 64)`.  
- These serve as inputs to all attention mechanisms.  



### **Step 2: Initialize Attention Mechanisms**
- Create objects for:
  - `AdditiveAttention` (requires query_dim, key_dim, attention_dim).  
  - `GeneralMultiplicativeAttention` (requires query_dim, key_dim).  
  - `ConcatMultiplicativeAttention` (requires query_dim, key_dim, attention_dim).  



### **Step 3: Apply Each Attention Mechanism**
- Pass the same **query, keys, and values** into all mechanisms:  
  - **Additive Attention** → produces `(context_add, weights_add)`.  
  - **General Multiplicative Attention** → produces `(context_gen, weights_gen)`.  
  - **Concat Multiplicative Attention** → produces `(context_con, weights_con)`.  

Each produces:  
- **context** → weighted sum of values (representation).  
- **attention_weights** → distribution over keys (importance of each key).  



### **Step 4: Print Attention Weights**
- Compare the **attention weight distributions** from each mechanism for the first batch.  
- This shows how each method assigns focus to different keys.  



### **Step 5: Dot-Product Attention**
- Expand the query to match sequence length → shape `(batch_size, seq_len, query_dim)`.  
- Apply **dot-product attention** using `dot_product_attention(Q, K, V)`.  
- Extract attention weights for the first query position.  



## Final Outcome
- You get attention weights from **all four mechanisms** side-by-side.  
- This allows direct comparison of how **different scoring functions** (additive, multiplicative, concat, dot-product) affect the distribution of attention.  

